# ezTrack — Location Tracking (single video)

Track one animal across a single video: load it, crop, exclude regions, build a background reference, (optionally) define regions of interest and a real-world scale, then track and summarize.

Every selection you make below (crop, mask, ROIs, scale) is stored on `session.selections`, a plain serializable object. You can `session.selections.save(...)` and reload it later to replay the exact same analysis — see step 9.

## 1. Load packages

In [ ]:
import holoviews as hv
import eztrack as ez

hv.extension("bokeh")

## 2. Point at the video
`dpath` is the folder, `file` the video name. `start`/`end` are frame numbers (`end=None` runs to the end). `region_names` names the ROIs you'll draw in step 6 (leave it `None` if you don't want any). On Windows, prefix raw paths with `r`, e.g. `r"C:\Videos"`.

In [ ]:
session = ez.Session(
    dpath="../../PracticeVideos/",
    file="LocationTracking_Clip.mp4",
    start=0,
    end=None,
    downsample=1.0,
    region_names=["left", "right", "top", "bottom"],
)

## 3. Load video and crop (optional)
Run the cell, then use the box-select tool beneath the image to draw the region to **keep**. Re-run this cell (and the steps after it) if you change the crop.

In [ ]:
hv.output(size=50)

ez.crop_tool(session)

## 4. Exclude internal regions (optional)
Draw any regions to **exclude** from analysis (e.g. a cable port). Double-click to start a polygon, single-click to add vertices, double-click to close.

In [ ]:
hv.output(size=100)

ez.mask_tool(session)

## 5. Build the reference (background) frame
The reference is the per-pixel **median** of frames sampled across the session, which removes the moving animal. Works well when the animal isn't still for >50% of the session.

**Alternative:** if you have an animal-free clip of the same arena, set `session.altfile = "empty.avi"` and call `ez.reference_frame(session, num_frames=50, altfile=True, frames=[0])`.

In [ ]:
hv.output(size=100)

ez.reference_frame(session, num_frames=50)
ez.image(session.reference, session.stretch, "Reference Frame", colorbar=True)

## 6. Define regions of interest (optional)
Draw the regions named in step 2, **in that order**. Overlapping regions are fine.

In [ ]:
hv.output(size=100)

ez.roi_tool(session)

## 7. Define a real-world scale (optional)
Click two points a known distance apart, then record that distance and its unit. Tracking output will include a `distance_<unit>` column.

In [ ]:
hv.output(size=100)

ez.distance_tool(session)

In [ ]:
ez.set_scale(session, real_distance=100, unit="cm")

## 8. (Optional) Save the selections
Persist crop + mask + ROIs + scale to a JSON file so this analysis can be replayed exactly (and reused across videos).

In [ ]:
session.selections.save("selections.json")
# later, or in another notebook:
# session.selections = ez.Selections.load("selections.json")

## 9. Tune tracking parameters
`threshold_pct`: keep only the brightest X% of frame-vs-reference differences (higher = stricter). `method`: `"abs"` (any change), `"light"` (animal lighter than background) or `"dark"` (animal darker). `window`: bias the estimate toward the previous location to ignore distractors (set `window=None` to disable).

The preview shows tracking on a few random frames — each original frame next to its thresholded difference, with the detected center of mass marked.

In [ ]:
params = ez.TrackParams(
    threshold_pct=99.5,
    method="abs",
    window=ez.Window(size=100, weight=0.9),
)

hv.output(size=100)
ez.threshold_preview(session, params, examples=4)

## 10. Track location

In [ ]:
location = ez.track(session, params)
location.head()

In [ ]:
location.to_csv("LocationOutput.csv", index=False)

## 11. Visualize
Motion trace (path overlaid on the reference, with ROI outlines) and a smoothed occupancy heatmap.

In [ ]:
hv.output(size=100)

ez.trace(session, location) + ez.heatmap(session, location)

## 12. Summarize
Total distance travelled and the proportion of time spent in each ROI. Pass `bins` to split the session into named frame ranges, e.g. `{"first_half": (0, 4500), "second_half": (4501, 9000)}`.

In [ ]:
summary = ez.summarize(location, session, bins=None)
summary

## 13. (Optional) Play back the tracking
Replays a segment inline with the estimated location marked. Set `save=True` to also write `video_output.avi`.

In [ ]:
ez.play_inline(
    session,
    ez.PlayParams(start=0, stop=200, fps=30, save=False),
    location,
)